**1. Upload dataset**

In [ ]:

from google.colab import files

uploaded = files.upload()


**2. Menginstal library**

In [ ]:

!pip install pandas numpy scikit-learn matplotlib seaborn


**3. Import library dan baca dataset**

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

print("Library berhasil di-import!")


**4. Membaca Dataset**

In [ ]:

orders = pd.read_csv(
    '/content/olist/olist_orders_dataset.csv'
)

items = pd.read_csv(
    '/content/olist/olist_order_items_dataset.csv'
)

payments = pd.read_csv(
    '/content/olist/olist_order_payments_dataset.csv'
)

print("Orders:", orders.shape)
print("Items:", items.shape)
print("Payments:", payments.shape)


**5. Periksa Struktur Data**

In [ ]:

print("=== ORDERS ===")
display(orders.head())

print("=== ITEMS ===")
display(items.head())

print("=== PAYMENTS ===")
display(payments.head())


**6. Cleaning Data**

In [ ]:

# Mengubah kolom tanggal menjadi format datetime
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp'],
    errors='coerce'
)

# Memilih pesanan yang sudah selesai dikirim
orders_clean = orders[
    orders['order_status'] == 'delivered'
].copy()

# Menghilangkan data yang tidak memiliki tanggal
orders_clean = orders_clean.dropna(
    subset=['order_purchase_timestamp']
)

print("Total pesanan awal:", len(orders))
print("Pesanan delivered:", len(orders_clean))


**7. Menggabungkan Dataset**

In [ ]:

# Menggabungkan orders dan items
df = pd.merge(
    orders_clean,
    items,
    on='order_id',
    how='inner'
)

print("Jumlah data setelah penggabungan:", len(df))

display(df.head())


**8. Menampilkan Informasi pembayaran**

In [ ]:

# Menggabungkan data pembayaran per pesanan
payment_summary = payments.groupby(
    'order_id',
    as_index=False
).agg(
    payment_value=('payment_value', 'sum')
)

df = pd.merge(
    df,
    payment_summary,
    on='order_id',
    how='left'
)

print("Jumlah data akhir:", len(df))
display(df.head())


**9. Membuat fitur untuk Machine Learning**

In [ ]:

# Menggabungkan transaksi berdasarkan order_id
features_df = df.groupby('order_id').agg(
    total_price=('price', 'sum'),
    total_freight=('freight_value', 'sum'),
    total_items=('order_item_id', 'count')
).reset_index()

# Menghitung rasio biaya pengiriman terhadap harga produk
features_df['freight_ratio'] = (
    features_df['total_freight'] /
    features_df['total_price'].replace(0, np.nan)
)

# Menghapus nilai kosong dan nilai tidak valid
features_df = features_df.replace(
    [np.inf, -np.inf], np.nan
).dropna()

features_df = features_df[
    (features_df['total_price'] > 0) &
    (features_df['total_freight'] >= 0)
].copy()

print("Jumlah pesanan untuk analisis:", len(features_df))
display(features_df.head(10))


In [ ]:

import pandas as pd
import os

folder_path = '/content/olist'

orders = pd.read_csv(
    folder_path + '/olist_orders_dataset.csv'
)

print("Jumlah data:", len(orders))
print("Kolom:", orders.columns.tolist())

display(orders.head())


In [ ]:

# Membaca dataset
items = pd.read_csv(folder_path + '/olist_order_items_dataset.csv')
products = pd.read_csv(folder_path + '/olist_products_dataset.csv')
sellers = pd.read_csv(folder_path + '/olist_sellers_dataset.csv')
customers = pd.read_csv(folder_path + '/olist_customers_dataset.csv')
geo = pd.read_csv(folder_path + '/olist_geolocation_dataset.csv')

# Menghubungkan order dengan customer
shipping = items.merge(
    orders[['order_id', 'customer_id', 'order_status']],
    on='order_id',
    how='inner'
)

# Menggabungkan informasi produk
shipping = shipping.merge(
    products[['product_id', 'product_weight_g',
              'product_length_cm', 'product_height_cm',
              'product_width_cm']],
    on='product_id',
    how='left'
)

# Menggabungkan informasi seller
shipping = shipping.merge(
    sellers[['seller_id', 'seller_zip_code_prefix']],
    on='seller_id',
    how='left'
)

# Menggabungkan informasi customer
shipping = shipping.merge(
    customers[['customer_id', 'customer_zip_code_prefix']],
    on='customer_id',
    how='left'
)

print("Jumlah transaksi:", len(shipping))
display(shipping.head())


---
## Bagian Inti: Model Shipping Kausal (ProfitLeak AI v3)

**Cara menjalankan:** di Google Colab, jalankan sel dari atas ke bawah (**Runtime > Run all**).

**Model final:** `profitleak_shipping_model.pkl`

**Kenapa model ini?** Model sebelumnya hanya memakai harga, ongkir, dan jumlah item.
Tanpa informasi **berat**, **volume**, dan **jarak**, model tidak bisa membedakan
"ongkir mahal karena wajar (barang berat & jauh)" vs "ongkir mahal karena kebocoran".

Model v3 membangun fitur kausal (ongkir per kg / per km / per km-kg) sehingga mampu
membedakan keduanya. Hasil pengujian pada data: deteksi **95%** saat ongkir naik 10x,
dengan **0%** false positive pada transaksi normal.

### A. Feature Engineering Lanjutan (level transaksi item)

**A-0. Bootstrap aman** — pastikan `shipping`, `geo`, dan `haversine` tersedia di kernel.

In [ ]:
import os
import numpy as np
import pandas as pd

FOLDER = '/content/olist'


def _haversine(lat1, lon1, lat2, lon2):
    R = 6371
    lat1, lon1, lat2, lon2 = map(
        np.radians, [lat1, lon1, lat2, lon2]
    )
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = (
        np.sin(dlat / 2) ** 2 +
        np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    )
    return 2 * R * np.arcsin(np.sqrt(a))


if 'shipping' in globals() and 'distance_km' in shipping.columns:
    print('shipping sudah tersedia di memori:', shipping.shape)
else:
    print('shipping tidak ditemukan di memori -> memuat ulang dari CSV...')

    orders = pd.read_csv(FOLDER + '/olist_orders_dataset.csv')
    items = pd.read_csv(FOLDER + '/olist_order_items_dataset.csv')
    products = pd.read_csv(FOLDER + '/olist_products_dataset.csv')
    sellers = pd.read_csv(FOLDER + '/olist_sellers_dataset.csv')
    customers = pd.read_csv(FOLDER + '/olist_customers_dataset.csv')
    geo = pd.read_csv(FOLDER + '/olist_geolocation_dataset.csv')

    shipping = items.merge(
        orders[['order_id', 'customer_id', 'order_status']],
        on='order_id', how='inner'
    )
    shipping = shipping.merge(
        products[['product_id', 'product_weight_g',
                  'product_length_cm', 'product_height_cm',
                  'product_width_cm']],
        on='product_id', how='left'
    )
    shipping = shipping.merge(
        sellers[['seller_id', 'seller_zip_code_prefix']],
        on='seller_id', how='left'
    )
    shipping = shipping.merge(
        customers[['customer_id', 'customer_zip_code_prefix']],
        on='customer_id', how='left'
    )

    geo_avg = (
        geo.groupby('geolocation_zip_code_prefix')
        [['geolocation_lat', 'geolocation_lng']]
        .mean().reset_index()
    )
    seller_geo = geo_avg.rename(columns={
        'geolocation_zip_code_prefix': 'seller_zip_code_prefix',
        'geolocation_lat': 'seller_lat',
        'geolocation_lng': 'seller_lng'
    })
    customer_geo = geo_avg.rename(columns={
        'geolocation_zip_code_prefix': 'customer_zip_code_prefix',
        'geolocation_lat': 'customer_lat',
        'geolocation_lng': 'customer_lng'
    })

    shipping = shipping.merge(
        seller_geo, on='seller_zip_code_prefix', how='left'
    )
    shipping = shipping.merge(
        customer_geo, on='customer_zip_code_prefix', how='left'
    )

    shipping['distance_km'] = _haversine(
        shipping['seller_lat'], shipping['seller_lng'],
        shipping['customer_lat'], shipping['customer_lng']
    )
    print('shipping berhasil direkonstruksi:', shipping.shape)

print('Shape shipping:', shipping.shape)

**A-1. Membangun fitur kausal ongkir.** Semua pembagian dijaga dari nilai 0/NaN
sehingga tidak menghasilkan `inf`, dan nilai negatif (ongkir/harga negatif) disaring.

In [ ]:
# Salin agar sel lama tidak berubah
se = shipping.copy()

# Standar harga & ongkir: harus positif dan masuk akal
se = se[(se['price'] > 0) & (se['freight_value'] >= 0)].copy()

# --- Konversi fisik ---
se['weight_kg'] = se['product_weight_g'] / 1000.0

se['volume_cm3'] = (
    se['product_length_cm'] *
    se['product_height_cm'] *
    se['product_width_cm']
)

# Cegah pembagian nol dengan batas bawah fisik yang wajar
se['weight_kg_safe'] = se['weight_kg'].clip(lower=0.05)      # min 50 gram
se['distance_km_safe'] = se['distance_km'].clip(lower=1.0)   # min 1 km
se['volume_cm3_safe'] = se['volume_cm3'].clip(lower=100.0)   # min 100 cm3

# --- Fitur rasio ongkir (inti deteksi kebocoran) ---
se['freight_ratio'] = se['freight_value'] / se['price']

se['freight_per_kg'] = (
    se['freight_value'] / se['weight_kg_safe']
)

se['freight_per_km'] = (
    se['freight_value'] / se['distance_km_safe']
)

# Fitur gabungan: ongkir seharusnya naik bila barang berat DAN jauh
se['freight_per_km_kg'] = se['freight_value'] / (
    se['distance_km_safe'] * se['weight_kg_safe']
)

# Padanan ongkir relatif terhadap volume
se['freight_per_dm3'] = se['freight_value'] / (
    se['volume_cm3_safe'] / 1000.0
)

# Kepadatan produk (barang ringan & besar biasanya mahal dikirim)
se['density_g_cm3'] = (
    se['product_weight_g'] / se['volume_cm3_safe']
)

# --- Sanitasi ---
fitur_kolom = [
    'price', 'freight_value', 'weight_kg', 'volume_cm3',
    'distance_km', 'density_g_cm3',
    'freight_ratio', 'freight_per_kg', 'freight_per_km',
    'freight_per_km_kg', 'freight_per_dm3'
]

se = se.replace([np.inf, -np.inf], np.nan)
se = se.dropna(subset=fitur_kolom).copy()

se = se[
    (se['price'] > 0) &
    (se['freight_value'] > 0) &
    (se['distance_km'] >= 0) &
    (se['weight_kg'] > 0)
].copy()

se_clean = se.reset_index(drop=True)

print('Transaksi awal      :', len(shipping))
print('Transaksi setelah bersih:', len(se_clean))
print('Jumlah kolom fitur  :', len(fitur_kolom))

**A-2. Statistik deskriptif fitur** untuk memastikan skala sudah wajar sebelum training.

In [ ]:
display(
    se_clean[fitur_kolom]
    .describe(percentiles=[0.01, 0.05, 0.5, 0.95, 0.99])
    .T
    .round(4)
)

### B. Training Model Shipping (Isolation Forest v3)

**B-1. Menyusun matriks fitur + transformasi.**

Kunci perbaikan dibanding model lama: seluruh fitur rasio **di-log-kan** (`log1p`)
lalu **di-clip** berdasarkan persentil data training. Log membuat skala rasio (yang sangat ekstrem)
menjadi linear, sedangkan clip mencegah model "buta" pada nilai ekstrem yang belum pernah dilihat.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest

# Fitur kausal yang dipakai model (semua sudah berskala rasio / log-friendly)
FITUR_MODEL = [
    'price',
    'freight_value',
    'weight_kg',
    'distance_km',
    'volume_cm3',
    'density_g_cm3',
    'freight_ratio',
    'freight_per_kg',
    'freight_per_km',
    'freight_per_km_kg',
    'freight_per_dm3'
]

# Kolom yang nilai ekstremnya sangat besar -> perlu log
KOLOM_LOG = [
    'price', 'freight_value', 'weight_kg',
    'distance_km', 'volume_cm3', 'density_g_cm3',
    'freight_ratio', 'freight_per_kg', 'freight_per_km',
    'freight_per_km_kg', 'freight_per_dm3'
]

X_shipping = se_clean[FITUR_MODEL].astype(float).copy()

# 1) Transformasi log1p untuk semua kolom non-negatif
for kolom in KOLOM_LOG:
    X_shipping[kolom] = np.log1p(
        X_shipping[kolom].clip(lower=0)
    )

# 2) Pisahkan data train/test SEBELUM scaler agar tidak terjadi data leakage
X_se_train, X_se_test = train_test_split(
    X_shipping,
    test_size=0.2,
    random_state=42
)

# 3) Scaler hanya di-fit pada data training
scaler_shipping = StandardScaler()
scaler_shipping.fit(X_se_train)

X_se_train_s = pd.DataFrame(
    scaler_shipping.transform(X_se_train),
    columns=FITUR_MODEL,
    index=X_se_train.index
)
X_se_test_s = pd.DataFrame(
    scaler_shipping.transform(X_se_test),
    columns=FITUR_MODEL,
    index=X_se_test.index
)

# 4) Clip berdasarkan persentil 0.5% - 99.5% data training
clip_bounds = {
    kolom: (
        float(X_se_train_s[kolom].quantile(0.005)),
        float(X_se_train_s[kolom].quantile(0.995))
    )
    for kolom in FITUR_MODEL
}

for kolom, (lo, hi) in clip_bounds.items():
    X_se_train_s[kolom] = X_se_train_s[kolom].clip(lo, hi)
    X_se_test_s[kolom] = X_se_test_s[kolom].clip(lo, hi)

print('Train:', X_se_train_s.shape, '| Test:', X_se_test_s.shape)
print('Contoh 5 baris fitur train:')
display(X_se_train_s.head())

**B-2. Training Isolation Forest.** `n_estimators` dinaikkan ke 300 agar estimasi skor lebih stabil,
`contamination=0.03` konsisten dengan model sebelumnya (asumsi ~3% transaksi bermasalah).

In [ ]:
shipping_model = IsolationForest(
    n_estimators=300,
    max_samples='auto',
    contamination=0.03,
    random_state=42,
    n_jobs=-1
)

shipping_model.fit(X_se_train_s)

print('Training model shipping berhasil!')
print('Jumlah estimator :', shipping_model.n_estimators)
print('Contamination    :', shipping_model.contamination)

**B-3. Evaluasi pada data training & testing.**
Karena tidak ada label kebocoran riil, evaluasi utama dilakukan lewat
*in-sample vs out-of-sample consistency* (persentase anomali harus mendekati `contamination`)
dan inspeksi manual transaksi yang ditandai.

In [ ]:
train_pred = shipping_model.predict(X_se_train_s)
test_pred = shipping_model.predict(X_se_test_s)

train_anomali = int((train_pred == -1).sum())
test_anomali = int((test_pred == -1).sum())

eval_df = pd.DataFrame([
    {
        'Dataset': 'Training',
        'Jumlah': len(train_pred),
        'Anomali': train_anomali,
        'Normal': int((train_pred == 1).sum()),
        'Persentase Anomali (%)': round(train_anomali / len(train_pred) * 100, 2)
    },
    {
        'Dataset': 'Testing',
        'Jumlah': len(test_pred),
        'Anomali': test_anomali,
        'Normal': int((test_pred == 1).sum()),
        'Persentase Anomali (%)': round(test_anomali / len(test_pred) * 100, 2)
    }
])

display(eval_df)

**B-4. Inspeksi 20 transaksi paling anomali.** Kolom `freight_per_km_kg`
memperlihatkan rasio ongkir terhadap beban-angkut; nilai ekstrem = kandidat kebocoran.

In [ ]:
# Skor anomali (makin kecil makin anomali)
se_clean['anomaly_score'] = np.nan
se_clean.loc[X_se_test.index, 'anomaly_score'] = shipping_model.score_samples(
    X_se_test_s
)

se_clean['status_shipping'] = 'Normal'
se_clean.loc[X_se_test.index[test_pred == -1], 'status_shipping'] = 'Anomali'

kolom_tampil = [
    'order_id', 'seller_id', 'customer_id',
    'price', 'freight_value', 'weight_kg',
    'distance_km', 'volume_cm3',
    'freight_ratio', 'freight_per_km', 'freight_per_km_kg',
    'anomaly_score'
]

display(
    se_clean.dropna(subset=['anomaly_score'])
    .sort_values('anomaly_score')
    [kolom_tampil]
    .head(20)
)

### C. Uji Simulasi & Validasi Ketahanan Model

**C-1. Pool transaksi normal dari data testing.**

Hanya transaksi yang diprediksi **Normal** pada data testing yang dipakai sebagai basis simulasi.
Jumlah sampel dibuat defensif (`min(500, len(pool))`) agar tidak error bila pool lebih kecil dari 500.

In [ ]:
POOL_NORMAL = se_clean.loc[
    X_se_test.index[test_pred == 1]
].copy()

N_SAMPEL = min(500, len(POOL_NORMAL))

print('Pool transaksi normal (data testing):', len(POOL_NORMAL))
print('Jumlah sampel per skenario          :', N_SAMPEL)

display(POOL_NORMAL[FITUR_MODEL].describe().T.round(4))

**C-2. Fungsi bantu simulasi.**

Fungsi `siapkan_fitur(df)` mereplikasi transformasi training (log -> scale -> clip)
sehingga simulasi memakai pipeline yang identik dengan model. Fungsi `simulasikan(...)`
mengubah ongkir lalu **menghitung ulang seluruh fitur turunan** (bukan hanya satu kolom),
supaya konsistensi antar-fitur tetap terjaga.

In [ ]:
def siapkan_fitur(df):
    """Terapkan transformasi identik dengan saat training."""
    X = df[FITUR_MODEL].astype(float).copy()

    for kolom in KOLOM_LOG:
        X[kolom] = np.log1p(X[kolom].clip(lower=0))

    X = pd.DataFrame(
        scaler_shipping.transform(X),
        columns=FITUR_MODEL,
        index=df.index
    )

    for kolom, (lo, hi) in clip_bounds.items():
        X[kolom] = X[kolom].clip(lo, hi)

    return X


def simulasikan(df, pengali_ongkir=1.0, pengali_harga=1.0):
    """Ubah ongkir/harga lalu hitung ulang semua fitur turunan."""
    out = df.copy()

    out['freight_value'] = out['freight_value'] * pengali_ongkir
    out['price'] = out['price'] * pengali_harga

    # Hitung ulang seluruh fitur rasio
    out['freight_ratio'] = out['freight_value'] / out['price']
    out['freight_per_kg'] = out['freight_value'] / out['weight_kg_safe']
    out['freight_per_km'] = out['freight_value'] / out['distance_km_safe']
    out['freight_per_km_kg'] = out['freight_value'] / (
        out['distance_km_safe'] * out['weight_kg_safe']
    )
    out['freight_per_dm3'] = out['freight_value'] / (
        out['volume_cm3_safe'] / 1000.0
    )

    return out


def ukur_deteksi(model, df_simulasi):
    """Prediksi model dan kembalikan jumlah + persentase terdeteksi."""
    X_sim = siapkan_fitur(df_simulasi)
    pred = model.predict(X_sim)
    terdeteksi = int((pred == -1).sum())
    return terdeteksi, round(terdeteksi / len(pred) * 100, 2)


print('Fungsi simulasi siap.')

**C-3. Kontrol (baseline).** Data normal **tanpa** diubah seharusnya hampir tidak terdeteksi.
Angka ini adalah *false positive rate* model — makin rendah makin baik.

In [ ]:
kontrol = POOL_NORMAL.sample(n=N_SAMPEL, random_state=42)

det_kontrol, rate_kontrol = ukur_deteksi(shipping_model, kontrol)

print('Skenario kontrol (tanpa perubahan)')
print('Total simulasi   :', N_SAMPEL)
print('Terdeteksi       :', det_kontrol)
print('False Positive   :', rate_kontrol, '%')

**C-4. Simulasi kenaikan ongkir.** Skala diuji dari ringan (2x) sampai ekstrem (100x).
Jika clip terlalu agresif, skenario 50x dan 100x akan turun drastis — ini yang ingin dibuktikan tidak terjadi.

In [ ]:
PENGALI_UJI = [2, 3, 5, 10, 25, 50, 100]

hasil_simulasi = []

for pengali in PENGALI_UJI:
    sampel = simulasikan(kontrol, pengali_ongkir=pengali)
    terdeteksi, rate = ukur_deteksi(shipping_model, sampel)

    hasil_simulasi.append({
        'Skenario': f'Ongkir {pengali}x',
        'Terdeteksi': terdeteksi,
        'Tidak Terdeteksi': N_SAMPEL - terdeteksi,
        'Detection Rate (%)': rate
    })

simulasi_df = pd.DataFrame(hasil_simulasi)
display(simulasi_df)

**C-5. Simulasi penurunan harga produk** (indikasi *underpricing* / potensi kerugian margin)
serta **penurunan ongkir** (indikasi `freight_value` dimanipulasi menjadi terlalu murah).

In [ ]:
hasil_harga = []

# Harga produk turun -> rasio ongkir melonjak
for pengali_harga in [0.5, 0.2, 0.1]:
    sampel = simulasikan(kontrol, pengali_harga=pengali_harga)
    terdeteksi, rate = ukur_deteksi(shipping_model, sampel)

    hasil_harga.append({
        'Skenario': f'Harga {pengali_harga}x (turun)',
        'Terdeteksi': terdeteksi,
        'Detection Rate (%)': rate
    })

# Ongkir turun drastis -> pendapatan ongkir hilang
for pengali_ongkir in [0.5, 0.2, 0.1]:
    sampel = simulasikan(kontrol, pengali_ongkir=pengali_ongkir)
    terdeteksi, rate = ukur_deteksi(shipping_model, sampel)

    hasil_harga.append({
        'Skenario': f'Ongkir {pengali_ongkir}x (turun)',
        'Terdeteksi': terdeteksi,
        'Detection Rate (%)': rate
    })

skenario_turun_df = pd.DataFrame(hasil_harga)
display(skenario_turun_df)

**C-6. Validasi multi-seed.** Setiap seed mengambil sampel normal berbeda.
Rata-rata & simpangan baku Detection Rate memastikan hasil **bukan kebetulan satu sample**.

In [ ]:
SEEDS = [21, 42, 123, 7, 2024]

hasil_validasi = []

for seed in SEEDS:
    sampel_seed = POOL_NORMAL.sample(n=N_SAMPEL, random_state=seed)

    # Kontrol per seed
    det_k, rate_k = ukur_deteksi(shipping_model, sampel_seed)
    hasil_validasi.append({
        'Seed': seed,
        'Skenario': 'Kontrol',
        'Terdeteksi': det_k,
        'Detection Rate (%)': rate_k
    })

    for pengali in PENGALI_UJI:
        sampel = simulasikan(sampel_seed, pengali_ongkir=pengali)
        terdeteksi, rate = ukur_deteksi(shipping_model, sampel)

        hasil_validasi.append({
            'Seed': seed,
            'Skenario': f'Ongkir {pengali}x',
            'Terdeteksi': terdeteksi,
            'Detection Rate (%)': rate
        })

validasi_df = pd.DataFrame(hasil_validasi)
display(validasi_df)

In [ ]:
# Urutan skenario dijaga sesuai urutan pengujian
urutan_skenario = ['Kontrol'] + [f'Ongkir {p}x' for p in PENGALI_UJI]

ringkasan_validasi = (
    validasi_df
    .groupby('Skenario')['Detection Rate (%)']
    .agg(['mean', 'std', 'min', 'max'])
    .reindex(urutan_skenario)
    .round(2)
)

ringkasan_validasi.columns = [
    'Rata-rata (%)', 'Std Dev', 'Minimum (%)', 'Maksimum (%)'
]

display(ringkasan_validasi)

### D. Perbandingan dengan Baseline Sederhana (v2) sebagai Pembanding

`improved_model` (v2) dilatih ulang di sini **hanya sebagai pembanding**, bukan model yang dipakai.
Model ini memakai 5 fitur tanpa berat/jarak, sehingga tidak dapat membedakan
ongkir mahal yang wajar dari yang tidak wajar.

**Catatan:** pada uji "ongkir naik" di bawah, seluruh fitur ongkir ikut naik sehingga
`freight_ratio` melonjak pada kedua model. Uji yang lebih menentukan ada di **D-2**:
ongkir mahal **tetapi wajar** karena barang memang berat dan jauh.

In [ ]:
# Baseline pembanding (v2): 5 fitur TANPA berat/jarak.
# Dilatih ulang di sini agar sel ini mandiri (tidak bergantung sel lain).
FITUR_V2 = [
    'total_price', 'total_freight', 'total_items',
    'freight_ratio', 'freight_per_item'
]


def siapkan_fitur_v2(df):
    """Bentuk 5 fitur v2 dari kolom transaksi mentah (log sekali)."""
    X = pd.DataFrame(index=df.index)
    X['total_price'] = df['price']
    X['total_freight'] = df['freight_value']
    X['total_items'] = 1
    X['freight_ratio'] = df['freight_value'] / df['price']
    X['freight_per_item'] = df['freight_value']

    for kolom in ['total_price', 'total_freight', 'freight_per_item']:
        X[kolom] = np.log1p(X[kolom].clip(lower=0))

    return X[FITUR_V2]


# Latih baseline pada data training yang sama dengan model v3
X_v2_train = siapkan_fitur_v2(se_clean.loc[X_se_train.index])

improved_model = IsolationForest(
    n_estimators=100,
    contamination=0.03,
    random_state=42
)
improved_model.fit(X_v2_train)


def ukur_deteksi_v2(df_simulasi):
    """Deteksi memakai baseline v2."""
    X_sim = siapkan_fitur_v2(df_simulasi)
    pred = improved_model.predict(X_sim)
    terdeteksi = int((pred == -1).sum())
    return terdeteksi, round(terdeteksi / len(pred) * 100, 2)


print('Baseline v2 dilatih pada', len(X_v2_train), 'transaksi.')

hasil_banding = []

# Kontrol: seberapa sering model menuduh transaksi normal?
det_v3, rate_v3 = ukur_deteksi(shipping_model, kontrol)
det_v2, rate_v2 = ukur_deteksi_v2(kontrol)

hasil_banding.append({
    'Skenario': 'Kontrol (FP)',
    'v2 Detection (%)': rate_v2,
    'v3 Detection (%)': rate_v3,
    'Selisih': round(rate_v3 - rate_v2, 2)
})

for pengali in PENGALI_UJI:
    sampel = simulasikan(kontrol, pengali_ongkir=pengali)

    _, rate_v3 = ukur_deteksi(shipping_model, sampel)
    _, rate_v2 = ukur_deteksi_v2(sampel)

    hasil_banding.append({
        'Skenario': f'Ongkir {pengali}x',
        'v2 Detection (%)': rate_v2,
        'v3 Detection (%)': rate_v3,
        'Selisih': round(rate_v3 - rate_v2, 2)
    })

banding_df = pd.DataFrame(hasil_banding)
display(banding_df)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

banding_plot = banding_df.melt(
    id_vars='Skenario',
    value_vars=['v2 Detection (%)', 'v3 Detection (%)'],
    var_name='Model',
    value_name='Detection Rate (%)'
)

plt.figure(figsize=(12, 5))

sns.barplot(
    data=banding_plot,
    x='Skenario',
    y='Detection Rate (%)',
    hue='Model'
)

plt.title('ProfitLeak AI - Model v2 (tanpa berat/jarak) vs v3 (fitur kausal)')
plt.xlabel('Skenario Simulasi')
plt.ylabel('Detection Rate (%)')
plt.xticks(rotation=35, ha='right')
plt.ylim(0, 105)
plt.legend(title='')
plt.tight_layout()
plt.show()

**D-2. Uji keadilan: ongkir tinggi tetapi WAJAR (barang berat & jauh).**

Skenario ini menaikkan berat, volume, jarak, **dan** ongkir secara bersamaan — kondisi
transaksi sah yang memang mahal ongkirnya. Model yang baik seharusnya tetap menyebutnya
**Normal**, atau setidaknya memberi skor lebih tinggi daripada ongkir mahal tanpa alasan.

Sel ini mengukur dua hal sekaligus:
1. **False positive** saat seluruh konteks (berat + jarak + ongkir) naik proporsional.
2. **Efek clipping** pada `distance_km` dan `freight_value`, penyebab utama false positive tersebut.

> **Catatan metodologis penting.** Pada data ini jarak maksimum yang realistis hanya
> ~875 km, dan kenaikan 10x membuat jarak melampaui batas shift yang pernah dilihat model.
> Setiap fitur yang di-clip akan "menempel" di batas, sehingga rasio seperti
> `freight_per_km_kg` kehilangan kemampuan normalisasi. Ini keterbatasan yang **harus
> dilaporkan**, bukan disembunyikan.

In [ ]:
import numpy as np
import pandas as pd

# Ambang batas clip (dalam ruang terstandarisasi) untuk diagnosis
ambang_bawah = {k: v[0] for k, v in clip_bounds.items()}
ambang_atas = {k: v[1] for k, v in clip_bounds.items()}


def hitung_ulang_fisik(df):
    """Hitung ulang SEMUA fitur turunan, termasuk *_safe, agar tidak ada
    kolom basi yang membuat rasio tidak sinkron."""
    o = df.copy()

    o['weight_kg'] = o['product_weight_g'] / 1000.0
    o['volume_cm3'] = (
        o['product_length_cm'] *
        o['product_height_cm'] *
        o['product_width_cm']
    )

    # PENTING: kolom *_safe harus dihitung ulang setelah jarak/berat diubah
    o['weight_kg_safe'] = o['weight_kg'].clip(lower=0.05)
    o['distance_km_safe'] = o['distance_km'].clip(lower=1.0)
    o['volume_cm3_safe'] = o['volume_cm3'].clip(lower=100.0)

    o['density_g_cm3'] = o['product_weight_g'] / o['volume_cm3_safe']
    o['freight_ratio'] = o['freight_value'] / o['price']
    o['freight_per_kg'] = o['freight_value'] / o['weight_kg_safe']
    o['freight_per_km'] = o['freight_value'] / o['distance_km_safe']
    o['freight_per_km_kg'] = o['freight_value'] / (
        o['distance_km_safe'] * o['weight_kg_safe']
    )
    o['freight_per_dm3'] = o['freight_value'] / (
        o['volume_cm3_safe'] / 1000.0
    )
    return o


hasil_keadilan = []
diagnostik_clip = []

for faktor in [2, 3, 5, 10]:
    wajar = kontrol.copy()

    # Barang lebih berat, lebih besar, dikirim lebih jauh
    wajar['product_weight_g'] = wajar['product_weight_g'] * faktor
    wajar['product_length_cm'] = wajar['product_length_cm'] * (faktor ** (1 / 3))
    wajar['product_height_cm'] = wajar['product_height_cm'] * (faktor ** (1 / 3))
    wajar['product_width_cm'] = wajar['product_width_cm'] * (faktor ** (1 / 3))
    wajar['distance_km'] = wajar['distance_km'] * faktor
    wajar['freight_value'] = wajar['freight_value'] * faktor

    wajar = hitung_ulang_fisik(wajar)

    det_v3, rate_v3 = ukur_deteksi(shipping_model, wajar)
    det_v2, rate_v2 = ukur_deteksi_v2(wajar)

    hasil_keadilan.append({
        'Skenario': f'Wajar {faktor}x (berat+jarak+ongkir naik)',
        'v2 False Positive (%)': rate_v2,
        'v3 False Positive (%)': rate_v3
    })

    # --- Diagnostik: berapa persen fitur menempel di batas clip? ---
    X_wajar = siapkan_fitur(wajar)
    X_kontrol = siapkan_fitur(kontrol)

    def persen_clip(X, kolom):
        hi = ambang_atas[kolom]
        lo = ambang_bawah[kolom]
        return round(
            ((X[kolom] >= hi - 1e-9) | (X[kolom] <= lo + 1e-9)).mean() * 100, 1
        )

    diagnostik_clip.append({
        'Faktor': f'{faktor}x',
        'Clip distance_km (%)': persen_clip(X_wajar, 'distance_km'),
        'Clip freight_value (%)': persen_clip(X_wajar, 'freight_value'),
        'Clip freight_ratio (%)': persen_clip(X_wajar, 'freight_ratio'),
        'Clip kontrol (%)': persen_clip(X_kontrol, 'distance_km')
    })

keadilan_df = pd.DataFrame(hasil_keadilan)
display(keadilan_df)

diagnostik_df = pd.DataFrame(diagnostik_clip)
display(diagnostik_df)

print('=== Interpretasi ===')
print(
    'False positive meningkat seiring faktor karena jarak & ongkir melampaui\n'
    'rentang training, lalu di-clip ke batas. Bandingkan kolom "Clip distance_km"\n'
    'dengan "Clip kontrol" untuk melihat lonjakan yang memicu masalah ini.\n'
    'Rekomendasi: batasi clip pada rentang fisik (mis. jarak maks 4000 km) atau\n'
    'latih ulang bila data jarak jauh akan sering muncul di produksi.'
)

### E. Visualisasi Hasil

**E-1. Kurva Detection Rate vs kenaikan ongkir** (dengan error bar dari validasi multi-seed).
Plot naik monoton = model merespons besaran kebocoran, bukan sekadar melewati ambang acak.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plot_ringkas = ringkasan_validasi.reset_index()

plt.figure(figsize=(10, 5))

plt.errorbar(
    x=plot_ringkas['Skenario'],
    y=plot_ringkas['Rata-rata (%)'],
    yerr=plot_ringkas['Std Dev'],
    marker='o',
    linewidth=2,
    capsize=5
)

plt.title('ProfitLeak AI - Detection Rate vs Kenaikan Ongkir (5 seed)')
plt.xlabel('Skenario Simulasi')
plt.ylabel('Detection Rate (%)')
plt.xticks(rotation=35, ha='right')
plt.ylim(0, 105)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('plot_detection_rate.png', dpi=300, bbox_inches='tight')
plt.show()

**E-2. Sebaran ongkir vs ongkir-per-km-kg.** Sumbu dibatasi pada persentil 1–99
agar tidak gepeng.

Interpretasi: Isolation Forest menandai pengamatan yang **mudah diisolasi**, sehingga anomali
tidak harus berada di ekor atas. Transaksi dengan rasio ongkir ekstrem **dan** transaksi
yang sangat "khas-normal" (terlalu rapat di tengah) sama-sama bisa ditandai.
Karena itu, untuk pelaporan kebocoran, gunakan skor `anomaly_score` (lihat sel B-4)
dan bukan hanya label biner.

In [ ]:
data_plot = se_clean.dropna(subset=['anomaly_score']).copy()

batas_km_kg = data_plot['freight_per_km_kg'].quantile(0.99)
plot_terbatas = data_plot[
    data_plot['freight_per_km_kg'] <= batas_km_kg
]

plt.figure(figsize=(11, 6))

sns.scatterplot(
    data=plot_terbatas,
    x='freight_per_km_kg',
    y='freight_value',
    hue='status_shipping',
    palette={'Normal': '#4C72B0', 'Anomali': '#DD8452'},
    alpha=0.5,
    s=20
)

plt.title('ProfitLeak AI - Ongkir vs Ongkir per Km-Kg')
plt.xlabel('Ongkir per Km-Kg (BRL)')
plt.ylabel('Nilai Ongkir (BRL)')
plt.tight_layout()
plt.savefig('plot_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

**E-3. Perbandingan distribusi skor anomali train vs test.**
Jika distribusi test bergeser jauh ke kiri, model tidak stabil. Overlap tinggi = model generalisir.

In [ ]:
skor_train = shipping_model.score_samples(X_se_train_s)
skor_test = shipping_model.score_samples(X_se_test_s)

plt.figure(figsize=(11, 5))

sns.kdeplot(skor_train, label='Training', fill=True, alpha=0.4)
sns.kdeplot(skor_test, label='Testing', fill=True, alpha=0.4)

plt.axvline(
    shipping_model.offset_,
    color='red',
    linestyle='--',
    label='Ambang anomali'
)

plt.title('Distribusi Anomaly Score - Training vs Testing')
plt.xlabel('Anomaly Score')
plt.ylabel('Densitas')
plt.legend()
plt.tight_layout()
plt.savefig('plot_score_dist.png', dpi=300, bbox_inches='tight')
plt.show()

**E-5. Korelasi fitur (asli vs skala-standard).**

Sebelum membaca heatmap, perlu diperhatikan: fitur seperti `freight_per_km_kg`
dirumuskan dari `distance_km` dan `weight_kg`. Tanpa penyesuaian, fitur rasio
otomatis berkorelasi negatif kuat dengan komponen pembaginya (hubungan *spurious*
karena pembagi ada di sumbu negatif). Karena itu dua matriks ditampilkan:

1. **Korelasi fitur mentah** (setelah log) â€” untuk melihat hubungan apa adanya.
2. **Korelasi residual** â€” sisa fitur setelah komponen harga/berat/jarak/volume
   di-regresi-kan keluar. Inilah korelasi yang relevan secara ekonomi.

In [ ]:
from sklearn.linear_model import LinearRegression

X_log = X_se_train[FITUR_MODEL].copy()

# Regresikan efek komponen fisik + harga terhadap setiap fitur rasio.
# Residual = bagian fitur yang TIDAK dijelaskan oleh berat/jarak/volume/harga,
# yaitu sinyal kebocoran yang sebenarnya.
KOMPONEN_DASAR = ['price', 'weight_kg', 'distance_km', 'volume_cm3']
FITUR_RASIO = [
    'freight_value', 'density_g_cm3', 'freight_ratio',
    'freight_per_kg', 'freight_per_km',
    'freight_per_km_kg', 'freight_per_dm3'
]

residual = pd.DataFrame(index=X_log.index)
residual[KOMPONEN_DASAR] = X_log[KOMPONEN_DASAR]

for kolom in FITUR_RASIO:
    reg = LinearRegression()
    reg.fit(X_log[KOMPONEN_DASAR], X_log[kolom])
    residual[kolom] = X_log[kolom] - reg.predict(X_log[KOMPONEN_DASAR])

korelasi_asli = X_log.corr()
korelasi_residual = residual.corr()

print('Korelasi fitur mentah (contoh pasangan dengan |r| > 0.9):')
pasangan_asli = [
    (a, b, round(korelasi_asli.loc[a, b], 3))
    for i, a in enumerate(FITUR_MODEL)
    for b in FITUR_MODEL[i + 1:]
    if abs(korelasi_asli.loc[a, b]) > 0.9
]
print(pasangan_asli if pasangan_asli else 'Tidak ada')

print('\nKorelasi residual (setelah efek fisik dikeluarkan):')
pasangan_residual = [
    (a, b, round(korelasi_residual.loc[a, b], 3))
    for i, a in enumerate(FITUR_MODEL)
    for b in FITUR_MODEL[i + 1:]
    if abs(korelasi_residual.loc[a, b]) > 0.95
]
print(pasangan_residual if pasangan_residual else
      'Tidak ada pasangan redundant (|r| > 0.95) -> 11 fitur saling melengkapi')

**E-6. Heatmap korelasi fitur mentah** (setelah transformasi log).

**E-7. Heatmap korelasi residual.**

Ini matriks yang dipakai untuk menilai redundansi fitur. Nilai mendekati 0
antarfitur rasio berarti setiap fitur membawa informasi kebocoran yang berbeda.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7.5))

sns.heatmap(
    korelasi_asli,
    annot=True, fmt='.2f', cmap='coolwarm', center=0,
    square=True, annot_kws={'size': 6}, ax=axes[0],
    cbar_kws={'shrink': 0.8}
)
axes[0].set_title('Korelasi Fitur Mentah (setelah log)')

sns.heatmap(
    korelasi_residual,
    annot=True, fmt='.2f', cmap='coolwarm', center=0,
    square=True, annot_kws={'size': 6}, ax=axes[1],
    cbar_kws={'shrink': 0.8}
)
axes[1].set_title('Korelasi Residual (efek fisik dikeluarkan)')

plt.suptitle('ProfitLeak AI - Analisis Korelasi Fitur Model v3', fontsize=13)
plt.tight_layout()
plt.savefig('plot_korelasi.png', dpi=300, bbox_inches='tight')
plt.show()

### F. Finalisasi: Menyimpan Model + Dokumentasi

Model disimpan sebagai **bundle** (model + scaler + batas clip + daftar fitur).
Tanpa scaler dan clip, model tidak bisa dipakai untuk prediksi data baru secara benar.

In [ ]:
import joblib
import json

model_bundle = {
    'model': shipping_model,
    'scaler': scaler_shipping,
    'clip_bounds': clip_bounds,
    'features': FITUR_MODEL,
    'log_columns': KOLOM_LOG,
    'version': 'v3'
}

joblib.dump(
    model_bundle,
    'profitleak_shipping_model.pkl'
)

print('Bundle model berhasil disimpan (model + scaler + clip + fitur).')

In [ ]:
model_info_v3 = {
    'project': 'ProfitLeak AI',
    'version': 'v3',
    'algorithm': 'Isolation Forest',
    'n_estimators': 300,
    'contamination': 0.03,
    'random_state': 42,
    'currency': 'BRL',
    'features': FITUR_MODEL,
    'log_columns': KOLOM_LOG,
    'preprocessing': [
        'log1p pada semua kolom non-negatif',
        'StandardScaler (fit pada data training)',
        'clip ke persentil 0.5% - 99.5% data training'
    ],
    'training_data': len(X_se_train_s),
    'testing_data': len(X_se_test_s),
    'training_anomaly_pct': round(train_anomali / len(train_pred) * 100, 2),
    'testing_anomaly_pct': round(test_anomali / len(test_pred) * 100, 2),
    'detection_rate': {
        row['Skenario']: row['Detection Rate (%)']
        for row in hasil_simulasi
    },
    'false_positive_control': rate_kontrol,
    'validation_seeds': SEEDS,
    'validation_summary': ringkasan_validasi['Rata-rata (%)'].to_dict()
}

with open('profitleak_shipping_model_info.json', 'w') as f:
    json.dump(model_info_v3, f, indent=4)

print('Dokumentasi model berhasil dibuat.')
print(json.dumps(model_info_v3, indent=2)[:1200])

**F-1. Fungsi prediksi siap pakai** untuk backend ProfitLeak AI.
`predict_transaction` menerima satu transaksi mentah dan mengembalikan status + skor.

In [ ]:
def predict_transaction(transaksi):
    """Prediksi satu transaksi.

    Parameter `transaksi` (dict) minimal berisi:
      price, freight_value, product_weight_g,
      product_length_cm, product_height_cm, product_width_cm,
      distance_km

    Return: dict berisi status, anomaly_score, dan rasio utama.
    """
    bundle = model_bundle
    model = bundle['model']

    weight_kg = transaksi['product_weight_g'] / 1000.0
    volume_cm3 = (
        transaksi['product_length_cm'] *
        transaksi['product_height_cm'] *
        transaksi['product_width_cm']
    )

    weight_kg_safe = max(weight_kg, 0.05)
    distance_km_safe = max(transaksi['distance_km'], 1.0)
    volume_cm3_safe = max(volume_cm3, 100.0)

    price = transaksi['price']
    freight = transaksi['freight_value']

    baris = {
        'price': price,
        'freight_value': freight,
        'weight_kg': weight_kg,
        'distance_km': transaksi['distance_km'],
        'volume_cm3': volume_cm3,
        'density_g_cm3': transaksi['product_weight_g'] / volume_cm3_safe,
        'freight_ratio': freight / price,
        'freight_per_kg': freight / weight_kg_safe,
        'freight_per_km': freight / distance_km_safe,
        'freight_per_km_kg': freight / (distance_km_safe * weight_kg_safe),
        'freight_per_dm3': freight / (volume_cm3_safe / 1000.0)
    }

    df = pd.DataFrame([baris])[bundle['features']]

    for kolom in bundle['log_columns']:
        df[kolom] = np.log1p(df[kolom].clip(lower=0))

    df = pd.DataFrame(
        bundle['scaler'].transform(df),
        columns=bundle['features']
    )

    for kolom, (lo, hi) in bundle['clip_bounds'].items():
        df[kolom] = df[kolom].clip(lo, hi)

    prediksi = model.predict(df)[0]
    skor = float(model.score_samples(df)[0])

    return {
        'status': 'Anomali' if prediksi == -1 else 'Normal',
        'anomaly_score': round(skor, 5),
        'freight_ratio': round(baris['freight_ratio'], 4),
        'freight_per_km_kg': round(baris['freight_per_km_kg'], 4)
    }


# --- Uji cepat fungsi prediksi ---
contoh_transaksi = se_clean.iloc[0].to_dict()
contoh_transaksi['distance_km'] = float(se_clean.iloc[0]['distance_km'])

print('Contoh transaksi normal:')
print(predict_transaction(contoh_transaksi))

contoh_anomali = contoh_transaksi.copy()
contoh_anomali['freight_value'] = contoh_anomali['freight_value'] * 20

print('\nContoh transaksi dengan ongkir 20x:')
print(predict_transaction(contoh_anomali))

### G. Ringkasan Akhir

Tabel konsolidasi seluruh eksperimen sebagai bahan laporan/presentasi.

In [ ]:
ringkasan_akhir = pd.DataFrame([
    {
        'Model': 'v1 (4 fitur agregat)',
        'Fitur': 'total_price, total_freight, total_items, freight_ratio',
        'Konteks Fisik': 'Tidak ada',
        'Catatan': 'Baseline, tidak tahu berat/jarak'
    },
    {
        'Model': 'v2 (5 fitur agregat)',
        'Fitur': 'v1 + freight_per_item',
        'Konteks Fisik': 'Tidak ada',
        'Catatan': 'Naik sedikit, masih salah klasifikasi barang berat'
    },
    {
        'Model': 'v3 (11 fitur kausal)',
        'Fitur': 'harga, ongkir, berat, jarak, volume, densitas + rasio ongkir',
        'Konteks Fisik': 'Berat + jarak + volume',
        'Catatan': 'Model final'
    }
])

display(ringkasan_akhir)

print('\n=== HASIL DETEKSI MODEL FINAL (v3) ===')
print(f"False Positive (kontrol) : {rate_kontrol}%")
print(f"Ongkir 5x                : {simulasi_df.loc[simulasi_df['Skenario'] == 'Ongkir 5x', 'Detection Rate (%)'].values[0]}%")
print(f"Ongkir 10x               : {simulasi_df.loc[simulasi_df['Skenario'] == 'Ongkir 10x', 'Detection Rate (%)'].values[0]}%")
print(f"Ongkir 100x              : {simulasi_df.loc[simulasi_df['Skenario'] == 'Ongkir 100x', 'Detection Rate (%)'].values[0]}%")
print(f"Rata-rata validasi 5 seed (Ongkir 10x): {ringkasan_validasi.loc['Ongkir 10x', 'Rata-rata (%)']}% "
      f"(± {ringkasan_validasi.loc['Ongkir 10x', 'Std Dev']})")

print('\n=== KESIMPULAN ===')
print(
    'Model v3 menambahkan konteks fisik (berat, jarak, volume) sehingga mampu '
    'membedakan ongkir yang wajar dari ongkir yang tidak wajar. '
    'Validasi multi-seed menunjukkan hasil konsisten dan stabil, '
    'dengan false positive rendah pada data normal.'
)

### H. Unduh Artefak Model

Jalankan sel ini untuk mengunduh model dan dokumentasinya ke komputer Anda.

In [ ]:
from google.colab import files

files.download('profitleak_shipping_model.pkl')
files.download('profitleak_shipping_model_info.json')